In [1]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="llama3.2:3b", temperature=0.2)

# result = model.invoke("What is the capital of India?")
# print(result)

In [2]:
# Set Token Limit
MAX_TOKENS = 150

In [3]:
from langgraph.graph import MessagesState

from langchain_core.messages.utils import trim_messages, count_tokens_approximately


# Function to call the model with trimmed messages
def call_model(state: MessagesState):
    # Trim conversation history -> last N messages that fit within the token budget
    messages = trim_messages(
        state["messages"],
        strategy="last",                      
        token_counter=count_tokens_approximately,
        max_tokens=MAX_TOKENS
    )

    print('Current Token Count ->', count_tokens_approximately(messages=messages))

    # Call the model and return the response
    for message in messages:
        print(message.content)

    response = model.invoke(messages)

    return {"messages": [response] }

In [4]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(MessagesState)

graph.add_node("call_model", call_model)
graph.add_edge(START, "call_model")
graph.add_edge("call_model", END)

In [5]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [6]:
config = {"configurable": {"thread_id": "chat-1"}}

result = workflow.invoke(
    {"messages": [{"role": "user", "content": "Hi, my name is Hemant."}]},
    config,
)

result["messages"][-1].content

Current Token Count -> 10
Hi, my name is Hemant.


"Hello Hemant! It's nice to meet you. Is there something I can help you with or would you like to chat?"

In [7]:
result = workflow.invoke(
    {"messages": [{"role": "user", "content": "What is my name?"}]},
    config,
)

result["messages"][-1].content

Current Token Count -> 49
Hi, my name is Hemant.
Hello Hemant! It's nice to meet you. Is there something I can help you with or would you like to chat?
What is my name?


'Your name is Hemant. We just established that earlier!'

In [8]:
result = workflow.invoke(
    {"messages": [{"role": "user", "content": "Can you explain short term memory?"}]},
    config,
)

result["messages"][-1].content

Current Token Count -> 81
Hi, my name is Hemant.
Hello Hemant! It's nice to meet you. Is there something I can help you with or would you like to chat?
What is my name?
Your name is Hemant. We just established that earlier!
Can you explain short term memory?


'Hemant, short-term memory refers to the ability to temporarily hold and process information in your mind for a short period of time. It\'s like a mental "holding area" where you can store information while you decide what to do with it.\n\nShort-term memory has a limited capacity and duration, typically lasting from a few seconds to a minute or two. Here are some key characteristics:\n\n1. **Limited capacity**: Short-term memory can hold only a small amount of information at a time.\n2. **Temporary storage**: Information is stored in short-term memory for a brief period before it\'s either retrieved, forgotten, or discarded.\n3. **Voluntary control**: You have conscious control over what information you focus on and when.\n\nWhen you hear new information, like your name (Hemant!), it enters your short-term memory. If you don\'t act on the information immediately, it may be lost unless you repeat it to yourself, write it down, or associate it with something else.\n\nShort-term memory i

In [9]:
result = workflow.invoke(
    {"messages": [{"role": "user", "content": "What is my name?"}]},
    config,
)

result["messages"][-1].content

Current Token Count -> 8
What is my name?


"I don't have any information about your identity or personal details, so I'm not aware of your name. We just started our conversation, and I'm here to help answer any questions you may have. How can I assist you today?"

In [14]:
for item in workflow.get_state({"configurable": {"thread_id": "chat-1"}}).values['messages']:
    print(item.content)
    print('-'*120)

Hi, my name is Hemant.
------------------------------------------------------------------------------------------------------------------------
Hello Hemant! It's nice to meet you. Is there something I can help you with or would you like to chat?
------------------------------------------------------------------------------------------------------------------------
I am learning LangGraph.
------------------------------------------------------------------------------------------------------------------------
LangGraph is a popular open-source library for building and optimizing neural networks in Python. It's designed to be highly customizable and flexible, making it a great choice for researchers and developers working on various NLP tasks.

What specific aspects of LangGraph are you currently learning about? Are you trying to build a model from scratch or exploring its capabilities with pre-built examples?
------------------------------------------------------------------------------